# ESM C Implementation (62% Train Spearman R, 43% Validation Spearman R)

Within each cycle of active learning:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have limited chances to make queries.

In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression

import os
from tqdm.auto import tqdm

from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

## 1. Collect Training Data

In [2]:
#with open('Hackathon_data/Hackathon_data/sequence.fasta', 'r') as f:
with open('sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [3]:
len(sequence_wt)

656

In [4]:
def get_mutated_sequence(mut, sequence_wt):
    '''
    Adds the specified mutation into the wild-type sequence.

    Params:
        mut (str): The mutation to be applied.
        sequence_wt (str): The wild-type sequence to which the mutation will be applied.
    Returns:
        A deep copy of the mutated sequence string.
    '''
    # wt - wild type; pos - position; mt - mutation
    wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

    sequence = deepcopy(sequence_wt)

    return sequence[:pos] + mt + sequence[pos+1:]

In [5]:
#df_train = pd.read_csv('Hackathon_data/Hackathon_data/train.csv')
df_train = pd.read_csv('train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [6]:
#df_test = pd.read_csv('Hackathon_data/Hackathon_data/test.csv')
df_test = pd.read_csv('test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [7]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a Prediction Model

### Hyperparameters

In [8]:
seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

### ProteinDataset
Convert amino acids to machine-readable data via embedding space.

In [9]:
def gen_emb_from_df(df, out_dir="esm_c_embeddings_variants", device="cuda:0", batch_size=8):
    '''
    Generate and cache ESM-C sequence embeddings for each unique mutant.
    Saves one mean-pooled embedding tensor per mutant to out_dir/{mutant}.pt.
    '''
    os.makedirs(out_dir, exist_ok=True)

    if isinstance(device, torch.device):
        use_cuda = device.type == "cuda"
    else:
        use_cuda = str(device).startswith("cuda")
    device = torch.device(device) if not isinstance(device, torch.device) else device

    # Each item is (name_for_file, sequence)
    data = [(m, s[:1000]) for m, s in df[["mutant", "sequence"]].drop_duplicates().values]
    print(f"Number of unique variants: {len(data)}")

    # Instantiate 600-million-parameter ESM-C model
    model = ESMC.from_pretrained("esmc_600m").to(device).eval()
    if use_cuda:
        model.half()  # Half-prec. to reduce GPU memory and runtime

    for i in tqdm(range(int(np.ceil(len(data) / batch_size)))):
        batch = data[i * batch_size:(i + 1) * batch_size]

        # Cache skip
        if all(os.path.exists(os.path.join(out_dir, f"{name}.pt")) for name, _ in batch):
            continue

        for name, sequence in batch:
            path = os.path.join(out_dir, f"{name}.pt")
            if os.path.exists(path):
                continue

            protein = ESMProtein(sequence=sequence)

            with torch.no_grad():
                protein_tensor = model.encode(protein)
                model_output = model.logits(
                    protein_tensor,
                    LogitsConfig(sequence=True, return_embeddings=True),
                )

            seq_emb = model_output.embeddings

            # Handle both [L, D] and [1, L, D] output shapes.
            if seq_emb.ndim == 3:
                seq_emb = seq_emb[0]
            seq_mean = seq_emb.float().mean(dim=0).detach().cpu()

            torch.save(seq_mean, path)

class DmsESMDataset(Dataset):
    def __init__(self, df, emb_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.emb_dir = emb_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        mutant = self.df.loc[idx, "mutant"]
        emb = torch.load(os.path.join(self.emb_dir, f"{mutant}.pt")).float()
        if self.is_train:
            y = torch.tensor(self.df.loc[idx, "DMS_score"], dtype=torch.float32)
            return emb, y
        return emb, torch.tensor(0.0, dtype=torch.float32)

In [10]:
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No CUDA device found; using CPU.")

# Run on GPU 0 when available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

True
NVIDIA GeForce RTX 5070 Laptop GPU


In [11]:
# Combine our datasets for embedding, them split them into train / test sets afterward. Each has mutated sequence variants stored as a separate .pt embedding.
all_df = pd.concat(
    [df_train[["mutant", "sequence"]], df_test[["mutant", "sequence"]]],
    ignore_index=True
).drop_duplicates("mutant")

gen_emb_from_df(all_df, out_dir="esm_c_embeddings_variants", device=device, batch_size=8)

Number of unique variants: 12464


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/1558 [00:00<?, ?it/s]

In [12]:
# gen_emb('Hackathon_Data/Hackathon_data/sequence.fasta', out_dir='esm_embeddings_train')
# gen_emb('Hackathon_Data/Hackathon_data/', out_dir='esm_embeddings_test')

In [13]:
# Use simple MLP model to predict from ESM C embeddings.
class MLPRegressor(nn.Module):
    def __init__(self, input_dim=1280, hidden_dim=640, dropout_p=0.1):
        super(MLPRegressor, self).__init__()

        # Only need to predict a single fitness score every time.
        output_dim = 1

        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid() # Ensures fitness scores in range (0,1).
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
'''
Params:
    model (torch.nn):               The untrained Fully-Connected Neural Network (FCN)
    train_dataset (DmsESMDataset):  The training dataset
    val_dataset (DmsESMDataset):    The validation dataset
    epochs (int):                   The number of epochs over which to train
    batch_size (int):               The number of batches in which to split the input data
    lr (float):                     The learning rate
    patience (int):                 The number of epochs to wait while the validation metric shows no improvement
    alpha (float):                  The Exponential Moving Average (EMA) weight by which the Spearman R is smoothed
    device (str):                   The CPU / GPU on which to run the model
Returns:
    model (torch.nn):               The trained FCN
    best_ckpt (dictionary):         Dictionary saving the best parameters
'''
def train_model_esm(model, train_dataset, val_dataset, epochs=100, batch_size=256, lr=1e-3, patience=10, alpha=0.3, device='cuda:0'):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Use MSE loss to handle bounded regression.
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

    best_val_spearman = -np.inf
    best_ckpt = None
    patience_counter = 0
    val_spearman_ema = None

    for epoch in range(epochs):
        model.train()
        train_losses = []
        train_preds = []
        train_targets = []

        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device).float()

            optimizer.zero_grad()
            outputs = model(inputs).squeeze(-1)
            loss = criterion(outputs, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, norm_type=2.0)
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(outputs.detach().cpu())
            train_targets.append(targets.detach().cpu())

        train_preds = torch.cat(train_preds).numpy()
        train_targets = torch.cat(train_targets).numpy()
        train_spearman = spearmanr(train_preds, train_targets).statistic

        # Validation
        model.eval()
        val_losses = []
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device).float()
                outputs = model(inputs).squeeze(-1)

                val_losses.append(criterion(outputs, targets).item())
                val_preds.append(outputs.detach().cpu())
                val_targets.append(targets.detach().cpu())

        val_preds = torch.cat(val_preds).numpy()
        val_targets = torch.cat(val_targets).numpy()
        val_spearman = spearmanr(val_preds, val_targets).statistic
        mean_train_loss = float(np.mean(train_losses))
        mean_val_loss = float(np.mean(val_losses))

        # Set moving average old value if it doesn't exist yet; prevents initial validation Spearman R from being artificially low.
        if val_spearman_ema == None:
            val_spearman_ema = val_spearman

        val_spearman_ema = alpha * val_spearman + (1 - alpha) * val_spearman_ema

        scheduler.step(val_spearman_ema)

        if np.isnan(train_spearman):
            train_spearman = 0.0
        if np.isnan(val_spearman):
            val_spearman = -1.0

        print(
            f"Epoch {epoch+1}: "
            f"Train Loss={mean_train_loss:.4f}, Train Spearman={train_spearman:.4f}, "
            f"Val Loss={mean_val_loss:.4f}, Val Spearman={val_spearman:.4f}, "
            f"LR={optimizer.param_groups[0]["lr"]}"
        )

        # Early stopping on validation Spearman (ranking quality)
        if val_spearman > best_val_spearman:
            best_val_spearman = val_spearman
            best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_ckpt is None:
        best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return model, best_ckpt

In [43]:
# Split data into train and validation sets, then form DmsESMDataset() classes from each.
df_train_split, df_val_split = train_test_split(
    df_train, test_size=val_ratio, random_state=seed, shuffle=True
)

train_ds = DmsESMDataset(df_train_split, "esm_c_embeddings_variants", is_train=True)
val_ds = DmsESMDataset(df_val_split, "esm_c_embeddings_variants", is_train=True)
test_ds = DmsESMDataset(df_test, "esm_c_embeddings_variants", is_train=False)

test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
# --------------- Train our model ---------------
model_esm = MLPRegressor(input_dim=1152,
                          hidden_dim=640,
                          dropout_p=0.3).to(device)
# TODO Currently using test values to make sure this works.
model_esm, best_ckpt_esm = train_model_esm(
    model_esm,
    train_ds,
    val_ds,
    epochs=500,  # 10
    batch_size=256,
    lr=5e-3,  # 1e-3
    patience=40,  # 5
    device=device,
 )
model_esm.load_state_dict(best_ckpt_esm)


# --------------- Test our model ---------------
model_esm.eval()
preds = []
with torch.no_grad():
    for sequences, _ in test_loader:
        # Inference on the test set
        sequences = sequences.to(device)
        outputs = model_esm(sequences)
        dms_score_pred = outputs.squeeze(-1).cpu().numpy()
        preds.extend(dms_score_pred.tolist())

df_pred = df_test.copy()
df_pred["DMS_score_predicted"] = preds

Epoch 1: Train Loss=0.1247, Train Spearman=0.0368, Val Loss=0.0916, Val Spearman=-0.0936, LR=0.005
Epoch 2: Train Loss=0.0717, Train Spearman=0.0550, Val Loss=0.0503, Val Spearman=0.0298, LR=0.005
Epoch 3: Train Loss=0.0652, Train Spearman=0.1482, Val Loss=0.0538, Val Spearman=0.0580, LR=0.005
Epoch 4: Train Loss=0.0618, Train Spearman=0.1990, Val Loss=0.0564, Val Spearman=0.0840, LR=0.005
Epoch 5: Train Loss=0.0562, Train Spearman=0.2290, Val Loss=0.0558, Val Spearman=0.1038, LR=0.005
Epoch 6: Train Loss=0.0533, Train Spearman=0.2561, Val Loss=0.0551, Val Spearman=0.1172, LR=0.005
Epoch 7: Train Loss=0.0492, Train Spearman=0.2855, Val Loss=0.0548, Val Spearman=0.1396, LR=0.005
Epoch 8: Train Loss=0.0461, Train Spearman=0.3117, Val Loss=0.0563, Val Spearman=0.1526, LR=0.005
Epoch 9: Train Loss=0.0441, Train Spearman=0.3295, Val Loss=0.0578, Val Spearman=0.1493, LR=0.005
Epoch 10: Train Loss=0.0422, Train Spearman=0.3363, Val Loss=0.0603, Val Spearman=0.1397, LR=0.005
Epoch 11: Train Lo

In [46]:
# Show current ESM-based predictions.
df_pred[['mutant', 'DMS_score_predicted']].head(n=10)

,mutant,DMS_score_predicted
0,V1D,0.517606
1,V1Y,0.671239
2,V1C,0.204429
3,V1A,0.527578
4,V1E,0.507166
5,V1W,0.308169
6,V1T,0.422611
7,V1R,0.323677
8,V1Q,0.470055
9,V1S,0.614710


In [47]:
# Save predictions to .csv.
df_pred[['mutant', 'DMS_score_predicted']].to_csv('test_predictions.csv', index=False)

In [48]:
group_number = "1"

with open("GroupName.txt", "w") as f:
    f.write(group_number.strip() + "\n")

print("Saved GroupName.txt")

Saved GroupName.txt


In [49]:
api_key = "df303a548bec1afcff7d7196650c5396cfa74c94a8be5357043602dcc7537ba7"

with open("APIKey.txt", "w") as f:
    f.write(api_key.strip() + "\n")

print("Saved APIKey.txt")

Saved APIKey.txt


## 3. Select Query for Next Round

In [50]:
# Show prediction distribution.
df_pred.sort_values('DMS_score_predicted', ascending=False)

,mutant,sequence,DMS_score_predicted
10966,D637K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,8.713370e-01
4741,P285K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,8.434619e-01
10384,T606K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,8.382987e-01
10375,T606Q,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,8.363634e-01
213,P12G,MVNEARGNSSLNGCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,8.307633e-01
...,...,...,...
3200,I181P,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,3.732108e-33
2674,L149D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,7.339934e-34
9925,W582P,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,3.354780e-34
2733,A154D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,3.146094e-34


In [51]:
# Write mutants with top ten DMS scores to .txt file.
df_top_ten = df_pred.sort_values('DMS_score_predicted', ascending=False).head(n=10)
df_top_ten.to_csv('top10.txt', columns=['mutant'], index=False, header=False)
print(df_top_ten)

      mutant                                           sequence  \
10966  D637K  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
4741   P285K  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
10384  T606K  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
10375  T606Q  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
213     P12G  MVNEARGNSSLNGCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
280     E15N  MVNEARGNSSLNPCLNGSASSGSESSKDSSRCSTPGLDPERHERLR...   
7206   S439N  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
10990  T638I  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
225     P12Y  MVNEARGNSSLNYCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
8994   L533Q  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   

       DMS_score_predicted  
10966             0.871337  
4741              0.843462  
10384             0.838299  
10375             0.836363  
213               0.830763  
280               0.818689  
7206              0.812668  
10990             0.812659  
2

In [52]:
# Example: randomly select 100 test variants to be queried.
# Note: Random selection may not be a good strategy
# TODO: Select query mutants for the next round based on your own criteria

queries = np.random.choice(df_test.mutant.values, size=100, replace=False)
queries

array(['D401M', 'K355V', 'K104M', 'S25Y', 'L210V', 'Q259T', 'Y118S',
       'G312D', 'Q508N', 'N619G', 'D187E', 'C130D', 'R356V', 'T450W',
       'V635N', 'Q647P', 'A84F', 'A208P', 'R40E', 'S429Q', 'D56F',
       'L125M', 'V573L', 'A586L', 'L61W', 'F515K', 'I279P', 'I111A',
       'E552W', 'P555C', 'E600F', 'R43H', 'I279A', 'D494S', 'K145A',
       'L346M', 'R43P', 'R369V', 'G498T', 'F514R', 'L149F', 'V268W',
       'Q277A', 'S393N', 'E299D', 'P571M', 'P66S', 'Y403P', 'I293D',
       'V635P', 'V430P', 'V634P', 'N11Q', 'Y375R', 'R518L', 'A352D',
       'E135H', 'K371H', 'S604G', 'Y188P', 'L524C', 'A291Q', 'V282A',
       'L641F', 'G161L', 'I478W', 'E46T', 'S368H', 'D91T', 'E46G',
       'V621C', 'N323A', 'R49E', 'V1T', 'L404S', 'S604L', 'V542I',
       'N629I', 'V449K', 'T548L', 'K408H', 'V268T', 'D584V', 'K272P',
       'S392K', 'T68R', 'K413Y', 'K355G', 'E325T', 'S17Y', 'H200R',
       'Q146T', 'K272W', 'S492R', 'E299C', 'F383C', 'S19C', 'W164I',
       'E379W', 'L399Y'], dtype=object

In [53]:
with open('query.txt', 'w') as f:
  for mutant in queries:
    f.write(mutant+'\n')